In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# SOURCE AND TARGET PATHS

source_file = f"{landing_folder_path}/drivers.json"
target_path = f"{s3_root_path}/Bronze/Aaisha/drivers"

print("Source File :", source_file)
print("Target Path :", target_path)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

drivers_schema = StructType([
    StructField("dateOfBirth", StringType()),
    StructField("driverId", StringType()),
    StructField("name", StructType([
        StructField("familyName", StringType()),
        StructField("givenName", StringType())
    ])),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
drivers_df = (
    spark.read
        .format("json")
        .option("multiLine", "true")
        .option("mode", "FAILFAST")
        .schema(drivers_schema)
        .load(source_file)
)

display(drivers_df)

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
(
    drivers_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)